In [99]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import scipy
import matplotlib as mpl
from scipy.special import gamma

STARTING_RANGE_PARAMETER = 0.00001 # In [fm^-2]
ENDING_RANGE_PARAMETER = 100
REDUCED_MASS = 935 * (4 / 5) # In [Mev / c^2], need to update value and units (10/11 A in MeV)
SUM_LIMIT = 100 # Determines the number of gaussians we expand our wave function to

V_LS = 11.71 # In MeV
DIFFUSIVITY = 0.6 # Diffusivity, may want to check the vaidity of this paticular number
r_0 = 1.2 # In fm, may want to chose a better value for small nuclei
A_C = 4 # The number of nucleons in the core

NUM_SUSY_GAUSSIANS = 15
SUSY_POTENTIAL_PARAMS = [0.015625,0.0192901,0.023815,0.0294012,0.0362978,0.0448121,0.0553235,0.0683007,0.0843218,0.104101,0.12852,0.158666,0.195884,0.241833,0.298559]
SUSY_POTENTIAL_COEFFS = [-912.425,2142.06,-409.428,-2018.96,47.0009,2144.66,335.787,-2156.45,-288.529,2091.06,-355.062,-1696.92,1654.04,-660.762,103.905]
CENTRAL_POTENTIAL_STRENGTH = -47.32 * 2

CENTRAL_POTENTIAL_PARAMETERS = [1 / ((2.30 * 2)**2)]

CENTRAL_MIXING_COEFFICIENTS = [1]

SPIN_ORBIT_POTENTIAL_PARAMETERS = [1 / (2.30**2)]

SPIN_ORBIT_MIXING_COEFFICIENTS = [1]

In [100]:
def single_particle_overlap(range_parameter_i, range_parameter_j, orbital_angular_momentum):
    term_1 = 2 * range_parameter_i * range_parameter_j
    term_2 = range_parameter_i**2 + range_parameter_j**2
    return (term_1 / term_2)**((5/2) + orbital_angular_momentum)

def single_particle_potential_element(range_parameter_i, range_parameter_j, orbital_angular_momentum,
                                      mixing_coefficient, potential_strength, potential_param):
    term_1 = 2 * range_parameter_i * range_parameter_i
    term_2 = range_parameter_i**2 + range_parameter_j**2 
    term_3 = range_parameter_i**2 * range_parameter_j**2 * potential_param
    return potential_strength * (term_1 / (term_2 + term_3))**((5/2) + orbital_angular_momentum)

def single_particle_kinetic_element(range_parameter_i, range_parameter_j, orbital_angular_momentum, μ=REDUCED_MASS):
    term_A = (2**((7/2) + orbital_angular_momentum)) / (3 + 2 * orbital_angular_momentum)
    term_1 = (orbital_angular_momentum * (orbital_angular_momentum + 1)) / (range_parameter_i * range_parameter_j)
    term_2 = ((range_parameter_i * range_parameter_j) / (range_parameter_i**2 + range_parameter_j**2))**(1.5 + orbital_angular_momentum)
    term_B = term_1 * term_2
    term_3 = (range_parameter_i * range_parameter_j)**(0.5 + orbital_angular_momentum)
    term_4 = (range_parameter_i**2 + range_parameter_j**2)**((7/2) + orbital_angular_momentum)
    term_5 = (orbital_angular_momentum + 1) * (orbital_angular_momentum + 2) * (range_parameter_i**4 + range_parameter_j**4)
    term_6 = (11 + 2 * orbital_angular_momentum * (5 + orbital_angular_momentum)) * range_parameter_i**2 * range_parameter_j**2
    term_C = (term_3 / term_4) * (term_5 - term_6)
    return (197**2 / (2 * μ)) * term_A * (term_B - term_C)

def r_minus_two_element(range_parameter_i, range_parameter_j, orbital_angular_momentum, μ=REDUCED_MASS):
    term_1 = (2**((9/2) + orbital_angular_momentum)) / ((3 + 2 * orbital_angular_momentum) * range_parameter_i * range_parameter_j)
    term_2 = ((range_parameter_i * range_parameter_j) / (range_parameter_i**2 + range_parameter_j**2))**((3/2) + orbital_angular_momentum)
    return (197**2 / (2 * μ)) * term_1 * term_2
    

In [101]:
def matrix_generation(orbital_angular_momentum, 
                      central_mixing_coefficients=CENTRAL_MIXING_COEFFICIENTS,
                      central_potential_parameters=CENTRAL_POTENTIAL_PARAMETERS,
                      susy_potential_mixing_coefficients=SUSY_POTENTIAL_COEFFS,
                      susy_potential_parameters=SUSY_POTENTIAL_PARAMS,
                      centeral_potential_strength=CENTRAL_POTENTIAL_STRENGTH,
                      size=SUM_LIMIT):
    h_matrix = np.zeros(shape=(size, size))
    n_matrix = np.zeros(shape=(size, size))

    for i in range(size):
        i_range_parameter = next_range_parameter(i)
        for j in range(size):
            j_range_parameter = next_range_parameter(j)
            kinetic_energy_term = single_particle_kinetic_element(i_range_parameter, j_range_parameter,
                                                                  orbital_angular_momentum)
            additional_susy_term = r_minus_two_element(i_range_parameter, j_range_parameter, orbital_angular_momentum)
            potential_energy_term = 0
            susy_energy_term = 0
            for k in range(len(central_mixing_coefficients)):
                potential_energy_term += single_particle_potential_element(
                    i_range_parameter, j_range_parameter, orbital_angular_momentum,
                    central_mixing_coefficients[k], centeral_potential_strength,
                    central_potential_parameters[k])
            for k in range(len(susy_potential_mixing_coefficients)):
                susy_energy_term += single_particle_potential_element(
                    i_range_parameter, j_range_parameter, orbital_angular_momentum,
                    susy_potential_mixing_coefficients[k], 1,
                    susy_potential_parameters[k])
            h_matrix[i, j] = kinetic_energy_term + potential_energy_term + susy_energy_term + additional_susy_term
            # h_matrix[j, i] = h_matrix[i, j]
            n_matrix[i, j] = single_particle_overlap(i_range_parameter, j_range_parameter, orbital_angular_momentum)

    return h_matrix, n_matrix

def next_range_parameter(i, starting_range_parameter=STARTING_RANGE_PARAMETER, ending_range_parameter=ENDING_RANGE_PARAMETER,
                         sum_limit=SUM_LIMIT):
    """
    Finds the next range parameter given the previous and initial range parameters.
    Currently using a simple geometric series to determine range parameters.
    Chose geometric basis parameters $\alpha_i = \alpha_1a^{i-1}$ with initial parameters $\alpha_1 = 0.01, a=2$

    Parameters
    ----------
    i : int detailing the iteration number

    Returns
    -------
    new_range_parameter: float

    """
    geometric_progression_number = (ending_range_parameter / starting_range_parameter)**(1 / (sum_limit - 1))
    new_range_parameter = starting_range_parameter * geometric_progression_number**(i)

    return new_range_parameter

In [103]:
s_h_matrix, s_n_matrix = matrix_generation(0)
s_eigenvalues, s_eigenvectors = scipy.linalg.eigh(s_h_matrix, s_n_matrix)
s_overlap_eigenvalues, s_overlap_eigenvectors = scipy.linalg.eigh(s_n_matrix)
s_overlap_matrix_condition_number = np.max(s_overlap_eigenvalues) / np.min(s_overlap_eigenvalues)
print(f"The s 1/2 overlap matrix condition number is {s_overlap_matrix_condition_number:e}")

s0_eigenvector = np.asmatrix(s_eigenvectors[:, 0])
s1_eigenvector = np.asmatrix(s_eigenvectors[:, 1])
print("The S state eigenvalues are", s_eigenvalues)

The s 1/2 overlap matrix condition number is 2.994260e+10
The S state eigenvalues are [-9.46322746e+02  2.54165896e-02  5.30794035e-02  1.41002857e-01
  3.07693469e-01  8.92021136e-01  2.48145417e+00  7.91485098e+00
  2.17763189e+01  6.01678850e+01  1.34077643e+02  2.70653721e+02
  4.14952591e+02  5.83852524e+02  7.33433186e+02  1.04058029e+03
  1.46632180e+03  2.18590331e+03  3.19076995e+03  4.69973948e+03
  6.81441718e+03  9.86173983e+03  1.41044261e+04  2.00691823e+04
  2.82948548e+04  3.96571872e+04  5.51653916e+04  7.62938241e+04
  1.04842265e+05  1.43294003e+05  1.94761854e+05  2.63403502e+05
  3.54487749e+05  4.74913988e+05  6.33452608e+05  8.41439357e+05
  1.11327318e+06  1.46739194e+06  1.92715472e+06  2.52225851e+06
  3.29019702e+06  4.27835642e+06  5.54633842e+06  7.16908885e+06
  9.24051293e+06  1.18781598e+07  1.52287624e+07  1.94752474e+07
  2.48451295e+07  3.16209738e+07  4.01530024e+07  5.08746536e+07
  6.43213770e+07  8.11536762e+07  1.02184953e+08  1.28415489e+08
  1.